In [ ]:
!gdown --folder https://drive.google.com/drive/folders/1BC6JqmBTdIisdcA8iEoir4n7Zvw2o8xX

Retrieving folder contents
Processing file 1oCUR9i3E-U1c_FKO3E9g_GSLqzOT59Ie all_results.csv
Processing file 1Tv9_F8Z7SFZ0WyqAyRFqDbSfgnh4vYcz results_maec16_vol_only.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1oCUR9i3E-U1c_FKO3E9g_GSLqzOT59Ie
To: /content/ecc-volatility-prediction-results-for-ablation/all_results.csv
100% 52.9k/52.9k [00:00<00:00, 48.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Tv9_F8Z7SFZ0WyqAyRFqDbSfgnh4vYcz
To: /content/ecc-volatility-prediction-results-for-ablation/results_maec16_vol_only.csv
100% 5.57k/5.57k [00:00<00:00, 14.0MB/s]
Download completed


In [ ]:
import pandas as pd
df_r = pd.read_csv('ecc-volatility-prediction-results-for-ablation/all_results.csv')
df_rv = pd.read_csv('ecc-volatility-prediction-results-for-ablation/results_maec16_vol_only.csv')

df1 = df_r.copy()

df1["experiment"] = df1["modality"].astype(str) + "_" + df1["fusion_type"].astype(str)

df2 = df_rv.copy()
df2["experiment"] = "volonly"

metrics = ["mse_3", "mse_7", "mse_15", "mse_30", "avg_mse"]

df1 = df1[["seed", "experiment"] + metrics]
df2 = df2[["seed", "experiment"] + metrics]

df_combined = pd.concat([df1, df2], ignore_index=True)

df = df_combined.pivot(
    index="seed",
    columns="experiment",
    values=metrics
)

df.columns = [
    f"{metric}_{experiment}"
    for metric, experiment in df.columns
]

df = df.reset_index()
df.columns.name = None

print(df.shape)
df.head()

(55, 41)


,seed,mse_3_audio_only_residual_gated,mse_3_chunk_text_audio_residual_gated,mse_3_text_audio_prompt1_residual_gated,mse_3_text_audio_prompt2_concat,mse_3_text_audio_prompt2_qa_only,mse_3_text_audio_prompt2_residual_gated,mse_3_text_only_residual_gated,mse_3_volonly,mse_7_audio_only_residual_gated,...,mse_30_text_only_residual_gated,mse_30_volonly,avg_mse_audio_only_residual_gated,avg_mse_chunk_text_audio_residual_gated,avg_mse_text_audio_prompt1_residual_gated,avg_mse_text_audio_prompt2_concat,avg_mse_text_audio_prompt2_qa_only,avg_mse_text_audio_prompt2_residual_gated,avg_mse_text_only_residual_gated,avg_mse_volonly
0,1,0.324385,0.325204,0.324586,0.365559,0.345481,0.324486,0.323839,0.343495,0.186803,...,0.146199,0.189645,0.218225,0.220556,0.218590,0.268114,0.244603,0.218323,0.217016,0.252745
1,3,0.324389,0.324242,0.324294,0.356013,0.398351,0.324280,0.329110,0.332479,0.175511,...,0.137441,0.151541,0.209179,0.209830,0.208893,0.263120,0.261448,0.208896,0.214190,0.226587
2,5,0.324746,0.325370,0.324835,0.334341,0.361698,0.324848,0.324944,0.337979,0.185811,...,0.131606,0.155664,0.208838,0.209878,0.208976,0.231344,0.234861,0.208978,0.209133,0.231468
3,6,0.341534,0.329290,0.328773,0.324387,0.407558,0.328731,0.332474,0.342854,0.190467,...,0.147492,0.176685,0.226295,0.215268,0.213648,0.234598,0.279255,0.213556,0.218454,0.237181
4,9,0.320261,0.320764,0.320070,0.329573,0.411001,0.319816,0.319180,0.337020,0.178463,...,0.132240,0.167958,0.209170,0.210688,0.208720,0.217021,0.279278,0.208273,0.207304,0.237377


In [3]:
import numpy as np
from scipy import stats

def summarize(df, winner, others):
    """Print mean MSE (sorted) and winner-vs-other paired-t results."""
    cols = [winner] + others
    print(f"n = {len(df)} seeds")
    print()
    print("Mean, sorted (lower = better):")
    print(df[cols].mean().sort_values())
    print()
    print(f"{'model':30s} {'mean diff':>12s} {'p-value':>10s}")
    for m in others:
        d = (df[m] - df[winner]).values
        t, p_two = stats.ttest_1samp(d, 0)
        p_one = p_two / 2 if t > 0 else 1 - p_two / 2
        print(f"{m:30s} {d.mean():12.5f} {p_one:10.4f}")

In [4]:
for h in [3, 7, 15, 30]:
    winner = f"mse_{h}_text_audio_prompt2_residual_gated"

    others = [
        f"mse_{h}_text_audio_prompt1_residual_gated",
        f"mse_{h}_audio_only_residual_gated",
        f"mse_{h}_text_only_residual_gated",
        f"mse_{h}_chunk_text_audio_residual_gated",
        f"mse_{h}_text_audio_prompt2_concat",
        f"mse_{h}_text_audio_prompt2_qa_only",
        f"mse_{h}_volonly",
    ]

    print(f"\n{'=' * 20} Horizon {h} {'=' * 20}")
    summarize(df, winner, others)


winner = "avg_mse_text_audio_prompt2_residual_gated"

others = [
    "avg_mse_text_audio_prompt1_residual_gated",
    "avg_mse_audio_only_residual_gated",
    "avg_mse_text_only_residual_gated",
    "avg_mse_chunk_text_audio_residual_gated",
    "avg_mse_text_audio_prompt2_concat",
    "avg_mse_text_audio_prompt2_qa_only",
    "avg_mse_volonly",
]

print("\n" + "=" * 20 + " Average MSE " + "=" * 20)
summarize(df, winner, others)


==================== Horizon 3 ====================
n = 55 seeds

Mean, sorted (lower = better):
mse_3_chunk_text_audio_residual_gated      0.331129
mse_3_text_audio_prompt2_residual_gated    0.332786
mse_3_text_only_residual_gated             0.334113
mse_3_audio_only_residual_gated            0.334666
mse_3_text_audio_prompt1_residual_gated    0.334727
mse_3_volonly                              0.340861
mse_3_text_audio_prompt2_concat            0.350897
mse_3_text_audio_prompt2_qa_only           0.386248
dtype: float64

model                             mean diff    p-value
mse_3_text_audio_prompt1_residual_gated      0.00194     0.1285
mse_3_audio_only_residual_gated      0.00188     0.2322
mse_3_text_only_residual_gated      0.00133     0.3038
mse_3_chunk_text_audio_residual_gated     -0.00166     0.8583
mse_3_text_audio_prompt2_concat      0.01811     0.0002
mse_3_text_audio_prompt2_qa_only      0.05346     0.0000
mse_3_volonly                       0.00807     0.0008

=========

### Pairwise comparison of all models

All-pairs paired one-sided t-test between models.

In [5]:
import numpy as np
import pandas as pd
from scipy import stats

model_suffixes = [
    "text_audio_prompt1_residual_gated",
    "text_audio_prompt2_residual_gated",
    "audio_only_residual_gated",
    "text_only_residual_gated",
    "chunk_text_audio_residual_gated",
    "text_audio_prompt2_concat",
    "text_audio_prompt2_qa_only",
    "volonly",
]

metrics = [f"mse_{h}" for h in [3, 7, 15, 30]] + ["avg_mse"]

for metric in metrics:
    all_models = [f"{metric}_{suffix}" for suffix in model_suffixes]

    short_names = {
        m: m.replace(f"{metric}_", "")
        for m in all_models
    }

    mean_diff = pd.DataFrame(index=all_models, columns=all_models, dtype=float)
    p_value = pd.DataFrame(index=all_models, columns=all_models, dtype=float)

    for row in all_models:
        for col in all_models:
            if row == col:
                mean_diff.loc[row, col] = 0.0
                p_value.loc[row, col] = np.nan
                continue

            # Positive means the row model has lower MSE (better)
            d = (df[col] - df[row]).values
            t, p_two = stats.ttest_1samp(d, 0, nan_policy="raise")
            p_one = p_two / 2 if t > 0 else 1 - p_two / 2

            mean_diff.loc[row, col] = d.mean()
            p_value.loc[row, col] = p_one

    mean_diff = mean_diff.rename(index=short_names, columns=short_names)
    p_value = p_value.rename(index=short_names, columns=short_names)

    print(f"\n{'=' * 25} {metric.upper()} {'=' * 25}")
    print(f"n = {len(df)} seeds\n")

    print("Mean difference (row model improvement over column model):")
    print(mean_diff.round(5))
    print()

    print("One-sided p-value (row model is better than column model):")
    print(p_value.round(4))


========================= MSE_3 =========================
n = 55 seeds

Mean difference (row model improvement over column model):
                                   text_audio_prompt1_residual_gated  \
text_audio_prompt1_residual_gated                            0.00000   
text_audio_prompt2_residual_gated                            0.00194   
audio_only_residual_gated                                    0.00006   
text_only_residual_gated                                     0.00061   
chunk_text_audio_residual_gated                              0.00360   
text_audio_prompt2_concat                                   -0.01617   
text_audio_prompt2_qa_only                                  -0.05152   
volonly                                                     -0.00613   

                                   text_audio_prompt2_residual_gated  \
text_audio_prompt1_residual_gated                           -0.00194   
text_audio_prompt2_residual_gated                            0.00000   
aud